In [1]:
import os
import json
from collections import defaultdict
from pathlib import Path
from pprint import pprint
import sys

dirpath_root = Path().resolve().parents[1]
sys.path.append(str(dirpath_root))

In [2]:
def extract_morphology_from_jsons_(cells_dir):
    # Dictionary to store groups of cell types with the same morphology
    morphology_groups = defaultdict(
        lambda: {"pop_names": [], "sections": set(), "section_lists": set()})
    
    # Iterate through all files in the cells/ directory
    for file_name in os.listdir(cells_dir):
        file_path = os.path.join(cells_dir, file_name)
        if file_name.endswith(".json"):
            try:
                with open(file_path, 'r') as f:
                    data = json.load(f)
                
                # Extract sections and section lists
                sections = set(data.get("secs", {}).keys())
                section_lists = set(data.get("secLists", {}).keys())
                
                # Group by morphology (sections and section lists)
                group_key = (frozenset(sections), frozenset(section_lists))
                pop = file_name.split('_')[0]
                morphology_groups[group_key]["pop_names"].append(pop)
                morphology_groups[group_key]["sections"].update(sections)
                morphology_groups[group_key]["section_lists"].update(section_lists)
                
            except Exception as e:
                print(f"Error reading {file_name}: {e}")

    # Convert group keys back to dict for readability
    grouped_morphologies = {}
    for idx, (key, value) in enumerate(morphology_groups.items(), 1):
        grouped_morphologies[f"group_{idx}"] = {
            "pop_names": value["pop_names"],
            "sections": list(value["sections"]),
            "section_lists": list(value["section_lists"]),
        }

    return grouped_morphologies

In [3]:
def _canon_seclists(secLists):
    """
    Return a hashable canonical representation of secLists:
      ( (listName, (sec1, sec2, ...)), ... )
    sorted by listName, and each list sorted.
    """
    if not isinstance(secLists, dict):
        return tuple()

    canon_items = []
    for lname, lst in secLists.items():
        # common patterns:
        #  - ["soma", "dend", ...]
        #  - {"secs": ["soma", ...]}  (seen in some exports)
        if isinstance(lst, dict) and "secs" in lst:
            lst = lst["secs"]

        if lst is None:
            secs = ()
        elif isinstance(lst, (list, tuple, set)):
            secs = tuple(sorted(map(str, lst)))
        else:
            # unexpected type -> keep a stable string
            secs = (str(lst),)

        canon_items.append((str(lname), secs))

    return tuple(sorted(canon_items, key=lambda x: x[0]))


def extract_morphology_from_jsons(cells_dir):
    morphology_groups = defaultdict(lambda: {
        "pop_names": [],
        "sections": set(),
        "section_lists": set(),
        "secLists_signature": None,   # store one exemplar
    })

    for file_name in os.listdir(cells_dir):
        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(cells_dir, file_name)
        try:
            with open(file_path, "r") as f:
                data = json.load(f)

            sections = set(data.get("secs", {}).keys())
            secLists = data.get("secLists", {}) or {}
            section_lists = set(secLists.keys())

            seclists_sig = _canon_seclists(secLists)

            # NEW: include seclists_sig in grouping key
            group_key = (frozenset(sections), seclists_sig)

            pop = file_name.split("_")[0]
            g = morphology_groups[group_key]
            g["pop_names"].append(pop)
            g["sections"].update(sections)
            g["section_lists"].update(section_lists)
            if g["secLists_signature"] is None:
                g["secLists_signature"] = seclists_sig

        except Exception as e:
            print(f"Error reading {file_name}: {e}")

    grouped = {}
    for idx, (_, value) in enumerate(morphology_groups.items(), 1):
        grouped[f"group_{idx}"] = {
            "pop_names": value["pop_names"],
            "sections": sorted(value["sections"]),
            "section_lists": sorted(value["section_lists"]),
            # optional: keep this to inspect exact list->secs mapping
            "secLists_signature": {
                lname: list(secs) for lname, secs in (value["secLists_signature"] or ())
            },
        }

    return grouped


In [4]:
# Example usage
cells_directory = dirpath_root / 'cells'  # Replace with the path to your cells/ directory
morph_data = extract_morphology_from_jsons(cells_directory)
print(json.dumps(morph_data, indent=4))
morph_data.pop('group_7')

{
    "group_1": {
        "pop_names": [
            "ITP4",
            "PT5B",
            "IT3",
            "IT2",
            "IT6",
            "IT5A",
            "IT5B",
            "CT5B",
            "CT5A",
            "CT6"
        ],
        "sections": [
            "Adend1",
            "Adend2",
            "Adend3",
            "Bdend",
            "axon",
            "soma"
        ],
        "section_lists": [
            "all",
            "apic",
            "apic_lowertrunk",
            "apic_trunk",
            "apic_tuft",
            "apic_uppertrunk",
            "dend_all",
            "proximal"
        ],
        "secLists_signature": {
            "all": [
                "Adend1",
                "Adend2",
                "Adend3",
                "Bdend",
                "soma"
            ],
            "apic": [
                "Adend1",
                "Adend2",
                "Adend3"
            ],
            "apic_lowertrunk": [
               

{'pop_names': ['bkgWeightPops.json'],
 'sections': [],
 'section_lists': [],
 'secLists_signature': {}}

In [5]:
from itertools import zip_longest

def print_groups_as_rows(grouped_morphologies):
    rows = []
    for g, v in grouped_morphologies.items():
        rows.append({
            "group": g,
            "pop_names": ", ".join(v["pop_names"]),
            "sections": ", ".join(v["sections"]),
            "section_lists": ", ".join(v["section_lists"]),
        })

    # column widths
    cols = rows[0].keys()
    widths = {
        c: max(len(c), max(len(r[c]) for r in rows))
        for c in cols
    }

    # header
    header = " | ".join(c.ljust(widths[c]) for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)

    # rows
    for r in rows:
        print(" | ".join(r[c].ljust(widths[c]) for c in cols))


print_groups_as_rows(morph_data)

group   | pop_names                                              | sections                                  | section_lists                                                                         
--------+--------------------------------------------------------+-------------------------------------------+---------------------------------------------------------------------------------------
group_1 | ITP4, PT5B, IT3, IT2, IT6, IT5A, IT5B, CT5B, CT5A, CT6 | Adend1, Adend2, Adend3, Bdend, axon, soma | all, apic, apic_lowertrunk, apic_trunk, apic_tuft, apic_uppertrunk, dend_all, proximal
group_2 | TC, RE, HTC                                            | soma                                      |                                                                                       
group_3 | NGF, TI, TI, TI                                        | dend, soma                                | all, dend_all, proximal                                                               
group_4 | 

In [6]:
import pandas as pd

df = pd.DataFrame([
    {
        "group": g,
        "pop_names": ", ".join(v["pop_names"]),
        "sections": ", ".join(v["sections"]),
        "section_lists": ", ".join(v["section_lists"]),
    }
    for g, v in morph_data.items()
])

#print(df.to_string(index=False))
df


,group,pop_names,sections,section_lists
0,group_1,"ITP4, PT5B, IT3, IT2, IT6, IT5A, IT5B, CT5B, C...","Adend1, Adend2, Adend3, Bdend, axon, soma","all, apic, apic_lowertrunk, apic_trunk, apic_t..."
1,group_2,"TC, RE, HTC",soma,
2,group_3,"NGF, TI, TI, TI","dend, soma","all, dend_all, proximal"
3,group_4,ITS4,"Adend1, Adend2, Adend3, Bdend, soma","all, apic, apic_lowertrunk, apic_trunk, apic_t..."
4,group_5,"SOM, PV","axon, dend, soma","all, dend_all, proximal"
5,group_6,VIP,"ori1, ori2, rad1, rad2, soma","all, dend_all, proximal"


In [7]:
def morph_groups_to_seclist_table(morph_data, joiner=", "):
    all_seclists = sorted({
        sl for g in morph_data.values()
        for sl in g.get("secLists_signature", {}).keys()
    })

    rows = []
    idx = []

    for group_name, g in morph_data.items():
        sig = g.get("secLists_signature", {}) or {}

        row = {}
        row["pop_names"] = ", ".join(g.get("pop_names", []))  # NEW column

        for sl in all_seclists:
            secs = sig.get(sl)
            row[sl] = "" if not secs else joiner.join(secs)

        rows.append(row)
        idx.append(group_name)

    df = pd.DataFrame(rows, index=idx)
    df.index.name = "pop_group"

    # ensure pop_names is the first column
    cols = ["pop_names"] + [c for c in df.columns if c != "pop_names"]
    df = df[cols]

    return df

df = morph_groups_to_seclist_table(morph_data)
df

,pop_names,all,apic,apic_lowertrunk,apic_trunk,apic_tuft,apic_uppertrunk,dend_all,proximal
pop_group,,,,,,,,,
group_1,"ITP4, PT5B, IT3, IT2, IT6, IT5A, IT5B, CT5B, C...","Adend1, Adend2, Adend3, Bdend, soma","Adend1, Adend2, Adend3",Adend1,"Adend1, Adend2",Adend3,Adend2,"Adend1, Adend2, Adend3, Bdend","Adend1, Bdend, soma"
group_2,"TC, RE, HTC",,,,,,,,
group_3,"NGF, TI, TI, TI","dend, soma",,,,,,dend,"dend, soma"
group_4,ITS4,"Adend1, Adend2, Adend3, Bdend, soma","Adend1, Adend2, Adend3",Adend1,"Adend1, Adend2",Adend3,Adend2,"Adend1, Adend2, Adend3, Bdend","Adend1, Bdend, soma"
group_5,"SOM, PV","dend, soma",,,,,,dend,"dend, soma"
group_6,VIP,"ori1, ori2, rad1, rad2, soma",,,,,,"ori1, ori2, rad1, rad2","ori1, rad1, soma"


In [8]:
import json
from typing import Dict, List, Optional
import pandas as pd
from pathlib import Path


def _find_matching_cell_rule(netParams, post_pop, cell_type, cell_model, pop_ynorm_range, cells_dir):
    """
    Find the matching cell rule for a population.
    
    Special handling for IT cells:
    - IT cells are differentiated by ynorm ranges (IT2, IT3, ITS4, ITP4)
    - Population has generic cellType='IT' but specific pop name
    - Must match: pop name OR (cellType + ynorm overlap)
    """
    model_suffix = cell_model.replace('HH_', '') if 'HH_' in cell_model else cell_model
    
    # Get cellParams dictionary
    cell_params_dict = netParams.get('cellParams', {})
    
    # Strategy 1: Try direct pop name match (e.g., ITS4_reduced)
    # This handles specific cell types like ITS4, and also IT2, IT3, etc.
    cell_rule_label = f"{post_pop}_{model_suffix}"
    cell_rule = cell_params_dict.get(cell_rule_label)
    
    if cell_rule is None:
        # Try loading pop-based from file
        json_path = Path(cells_dir) / f"{post_pop}_{model_suffix}_cellParams.json"
        if json_path.exists():
            try:
                with open(json_path, 'r') as f:
                    cell_rule = json.load(f)
                    # Found it - verify it's appropriate
                    if _verify_cell_rule_match(cell_rule, cell_type, cell_model, pop_ynorm_range, post_pop):
                        return cell_rule
                    else:
                        cell_rule = None  # Match failed, keep searching
            except Exception as e:
                print(f"Warning: Error loading {json_path}: {e}")
    
    # Strategy 2: Try cellType-based match (e.g., RE_reduced for IRE)
    # This handles cases where multiple pops share the same cellType
    if cell_rule is None:
        cell_rule_label = f"{cell_type}_{model_suffix}"
        cell_rule = cell_params_dict.get(cell_rule_label)
        
        if cell_rule is None:
            # Try loading cellType-based from file
            json_path = Path(cells_dir) / f"{cell_type}_{model_suffix}_cellParams.json"
            if json_path.exists():
                try:
                    with open(json_path, 'r') as f:
                        cell_rule = json.load(f)
                        # Found it - verify it's appropriate
                        if _verify_cell_rule_match(cell_rule, cell_type, cell_model, pop_ynorm_range, post_pop):
                            return cell_rule
                        else:
                            cell_rule = None  # Match failed
                except Exception as e:
                    print(f"Warning: Error loading {json_path}: {e}")
    
    # Verify final match
    if cell_rule is not None:
        if _verify_cell_rule_match(cell_rule, cell_type, cell_model, pop_ynorm_range, post_pop):
            return cell_rule
    
    return None


def _verify_cell_rule_match(cell_rule, cell_type, cell_model, pop_ynorm_range, post_pop):
    """
    Verify that a cell rule matches the population's requirements.
    
    Matching rules:
    1. cellModel must match (if specified in conds)
    2. cellType must match OR be compatible (IT cells special case)
    3. ynorm must overlap (if both specified)
    """
    conds = cell_rule.get('conds', {})
    
    # Check cellModel match (if specified in conds)
    cond_cell_model = conds.get('cellModel')
    if cond_cell_model is not None and cond_cell_model != cell_model:
        return False
    
    # Check cellType match
    cond_cell_type = conds.get('cellType')
    if cond_cell_type is not None:
        # Special case for IT cells: pop name might be IT2, IT3, ITS4, etc.
        # but conds.cellType could be 'IT' (generic) or specific (IT2, ITS4)
        # If conds specifies specific type (e.g., ITS4), pop must match
        # If pop is generic IT, accept if ynorm matches
        
        if cond_cell_type != cell_type:
            # Check if this is an IT cell variant
            if cell_type == 'IT' and cond_cell_type in ['IT', 'ITS4', 'ITP4']:
                # IT population with specific cell rule (e.g., ITS4_reduced)
                # This is OK - will be validated by ynorm overlap
                pass
            elif cond_cell_type == 'IT' and post_pop in ['IT2', 'IT3', 'ITS4', 'ITP4', 'IT5A', 'IT5B', 'IT6']:
                # Specific IT pop with generic IT cell rule
                # This is OK - will be validated by ynorm overlap
                pass
            else:
                # Genuine mismatch
                return False
    
    # Check ynorm overlap if both are specified
    cond_ynorm = conds.get('ynorm')
    if (cond_ynorm is not None and isinstance(cond_ynorm, (list, tuple)) and len(cond_ynorm) == 2 and
        pop_ynorm_range is not None and isinstance(pop_ynorm_range, (list, tuple)) and len(pop_ynorm_range) == 2):
        
        cond_min, cond_max = cond_ynorm
        pop_min, pop_max = pop_ynorm_range
        
        # Ranges must overlap
        if cond_max <= pop_min or cond_min >= pop_max:
            return False
    
    return True


def _expand_section_spec(sec_spec, secs_dict, sec_lists_dict):
    """
    Expand section specification to list of section names.
    
    Parameters
    ----------
    sec_spec : str or list
        Section specification (section name, list of names, or secList name)
    secs_dict : dict
        Dictionary of sections from cell rule
    sec_lists_dict : dict
        Dictionary of section lists from cell rule
        
    Returns
    -------
    tuple
        (list of section names, section_list_name or None)
    """
    section_names = []
    section_list_name = None
    
    if isinstance(sec_spec, str):
        # Check if it's a section list name
        if sec_spec in sec_lists_dict:
            section_names = sec_lists_dict[sec_spec]
            section_list_name = sec_spec
        # Check if it's a single section name
        elif sec_spec in secs_dict:
            section_names = [sec_spec]
        else:
            # Not found
            print(f"Warning: Section or section list '{sec_spec}' not found in cell rule")
            pass
    elif isinstance(sec_spec, list):
        # It's a list of section names
        section_names = sec_spec
    
    return section_names, section_list_name



def analyze_connections(netParams: Dict, cells_dir: str = 'cells') -> pd.DataFrame:
    """
    Analyze connections in netParams and return a table of valid connections.
    
    This function processes connection parameters to determine which connections
    are valid based on cell population positioning (ynorm ranges).
    
    Parameters
    ----------
    netParams : dict
        Network parameters dictionary (can be NetParams object or dict) containing:
        - connParams: Connection parameters
        - popParams: Population parameters (with ynormRange)
        - cellParams: Cell parameters (loaded via loadCellParamsRule)
        
    cells_dir : str, optional
        Directory containing cell JSON files (default: 'cells')
        
    Returns
    -------
    pd.DataFrame
        Table with columns:
        - conn_name: Name of the connection
        - pre_pops: Presynaptic populations (comma-separated)
        - post_pop: Postsynaptic population
        - ynorm: ynorm range from postConds if specified
        - sections: List of sections (comma-separated)
        - section_list_name: Name of section list if specified
    
    Raises
    ------
    Exception
        If a connection has more than one postsynaptic population.
    
    Notes
    -----
    Connection matching logic:
    - Populations have 'cellType', 'cellModel', and 'ynormRange'
    - Cell rules have 'conds' with 'cellType', 'cellModel', and 'ynorm'
    - Matching: pop.cellType matches cellRule.conds.cellType AND
                pop.cellModel matches cellRule.conds.cellModel AND
                pop.ynormRange overlaps with cellRule.conds.ynorm
    - ynorm in connParams.postConds filters CELLS by position, not sections
    - Connections are discarded if ynorm range doesn't overlap with pop's ynormRange
    """
    
    # Convert NetParams object to dict if needed
    if hasattr(netParams, 'todict'):
        netParams = netParams.todict()
    elif hasattr(netParams, '__dict__'):
        netParams = netParams.__dict__
    
    results = []
    
    # Get network size for normalization
    sizeY = netParams.get('sizeY', 2000.0)
    
    # Iterate through all connections in connParams
    for conn_name, conn_config in netParams.get('connParams', {}).items():
        
        # Step 2: Get postsynaptic population from postConds
        post_conds = conn_config.get('postConds', {})
        
        # Extract post population(s)
        post_pop = post_conds.get('pop')
        if post_pop is None:
            # Skip connections without explicit pop specification
            print(f"Warning: Connection '{conn_name}' has no postsynaptic population specified")
            continue
            
        # Check if more than one post population
        if isinstance(post_pop, list):
            if len(post_pop) > 1:
                raise Exception(
                    f"Connection '{conn_name}' has more than one postsynaptic "
                    f"population: {post_pop}"
                )
            post_pop = post_pop[0]
        
        # Get presynaptic population(s)
        pre_conds = conn_config.get('preConds', {})
        pre_pops = pre_conds.get('pop', [])
        if not isinstance(pre_pops, list):
            pre_pops = [pre_pops] if pre_pops else []
        
        # Step 3: Get description of postsynaptic cells
        pop_params = netParams.get('popParams', {}).get(post_pop, {})
        if not pop_params:
            print(f"Warning: Population '{post_pop}' not found in popParams")
            continue
            
        cell_type = pop_params.get('cellType')
        cell_model = pop_params.get('cellModel', 'HH_reduced')
        pop_ynorm_range = pop_params.get('ynormRange')
        
        if not cell_type:
            print(f"Warning: No cellType specified for population '{post_pop}'")
            continue
        
        # Find matching cell rule
        cell_rule = _find_matching_cell_rule(
            netParams, 
            post_pop, 
            cell_type, 
            cell_model, 
            pop_ynorm_range,
            cells_dir
        )
        
        if cell_rule is None:
            print(f"Warning: Could not find cell rule for population '{post_pop}'")
            continue
        
        # Step 4: Read 'sec' field from connection and find all sections
        sec_spec = conn_config.get('sec')
        if sec_spec is None:
            # Default to 'soma' if exists, else first available section
            sec_spec = 'soma' if 'soma' in cell_rule.get('secs', {}) else None
            if sec_spec is None and cell_rule.get('secs'):
                sec_spec = list(cell_rule['secs'].keys())[0]
        
        if sec_spec is None:
            print(f"Warning: No sections defined in cell rule for population '{post_pop}'")
            continue
        
        # Expand section specification
        section_names, section_list_name = _expand_section_spec(
            sec_spec, 
            cell_rule.get('secs', {}), 
            cell_rule.get('secLists', {})
        )
        
        if not section_names:
            print(f"Warning: No sections found for '{sec_spec}' in '{post_pop}' for conn '{conn_name}'")
            continue
        
        # Step 5: Check ynorm filtering
        ynorm = post_conds.get('ynorm')
        
        if ynorm is not None and isinstance(ynorm, (list, tuple)) and len(ynorm) == 2:
            # Check if this ynorm range overlaps with population's ynormRange
            if pop_ynorm_range and isinstance(pop_ynorm_range, (list, tuple)) and len(pop_ynorm_range) == 2:
                # Check for overlap: [a1, a2] overlaps [b1, b2] if NOT (a2 < b1 OR a1 > b2)
                ynorm_min, ynorm_max = ynorm
                pop_min, pop_max = pop_ynorm_range
                
                # No overlap - discard connection
                if ynorm_max <= pop_min or ynorm_min >= pop_max:
                    # Connection targets cells outside the population's range
                    continue
        
        # Format ynorm for display
        ynorm_str = ''
        if isinstance(ynorm, (list, tuple)) and len(ynorm) == 2:
            ynorm_str = f"[{ynorm[0]}, {ynorm[1]}]"
        elif ynorm is not None:
            ynorm_str = str(ynorm)
        
        # Step 6: Add to results
        results.append({
            'conn_name': conn_name,
            'pre_pops': ', '.join(pre_pops) if pre_pops else '',
            'post_pop': post_pop,
            'ynorm': ynorm_str,
            'sections': ', '.join(section_names),
            'section_list_name': section_list_name if section_list_name else ''
        })
    
    # Convert to DataFrame
    df = pd.DataFrame(results)
    
    # Reorder columns for better readability
    if not df.empty:
        column_order = ['conn_name', 'pre_pops', 'post_pop', 'ynorm', 
                        'sections', 'section_list_name']
        df = df[column_order]
    
    return df

In [9]:

fpath_par = ('/ddn/niknovikov19/repo/A1_OUinp/analysis/model_utils/test_data/netParams.json')

with open(fpath_par, 'r') as fid:
    net_par = json.load(fid)['net']['params']
df = analyze_connections(net_par)
df

,conn_name,pre_pops,post_pop,ynorm,sections,section_list_name
0,CxTh_CT5A_HTC,CT5A,HTC,,soma,
1,CxTh_CT5A_IRE,CT5A,IRE,,soma,
2,CxTh_CT5A_TC,CT5A,TC,,soma,
3,CxTh_CT5A_TI,CT5A,TI,,soma,
4,CxTh_CT5B_HTC,CT5B,HTC,,soma,
...,...,...,...,...,...,...
1540,frz_EE_PT5B_IT5B_5B,PT5Bfrz,IT5B,"[0.667, 0.775]","Adend1, Adend2, Adend3, Bdend",dend_all
1541,frz_EE_PT5B_IT6_6,PT5Bfrz,IT6,"[0.775, 1]","Adend1, Adend2, Adend3, Bdend",dend_all
1542,frz_EE_PT5B_ITP4_4,PT5Bfrz,ITP4,"[0.475, 0.625]","Adend1, Adend2, Adend3, Bdend",dend_all
1543,frz_EE_PT5B_ITS4_4,PT5Bfrz,ITS4,"[0.475, 0.625]","Adend1, Adend2, Adend3, Bdend",dend_all


In [10]:
import numpy as np

def _strip_frz(s):
    if not isinstance(s, str) or not s:
        return s
    if s.endswith('frz'):
        s = s[:-3]
    return s

df_ = df.copy()
df_['pre_pops'] = df_['pre_pops'].apply(_strip_frz)

n = np.asarray([len(x) for x in df_['pre_pops']])

idx = np.argsort(n)[::-1]
df_ = df_.iloc[idx].reset_index(drop=True)
df_

,conn_name,pre_pops,post_pop,ynorm,sections,section_list_name
0,PulseSeq->TC,PulseSeq,TC,,soma,
1,II_NGF5A_NGF4_4,NGF5A,NGF4,"[0.475, 0.625]","soma, dend",proximal
2,II_NGF5A_NGF3_3,NGF5A,NGF3,"[0.08, 0.475]","soma, dend",proximal
3,II_NGF5A_NGF2_2,NGF5A,NGF2,"[0.05, 0.08]","soma, dend",proximal
4,II_NGF5A_NGF1_1,NGF5A,NGF1,"[0.0, 0.05]","soma, dend",proximal
...,...,...,...,...,...,...
1540,ITh_TC_TC,TC,TC,,soma,
1541,ITh_TC_IREM,TC,IREM,,soma,
1542,ITh_TC_IRE,TC,IRE,,soma,
1543,ITh_TC_HTC,TC,HTC,,soma,


In [11]:
def match_subconns_to_df(netParams, df, debug=False):
    """
    Matches subConnParams to dataframe rows based on preConds/postConds.
    
    Parameters
    ----------
    netParams : dict
        Network parameters dictionary containing subConnParams
    df : pandas.DataFrame
        DataFrame with columns 'pre_pops' and 'post_pop'
    debug : bool
        If True, print detailed matching information
    
    Returns
    -------
    pandas.DataFrame
        Updated dataframe with 'subconn_name' and 'sec' columns
    
    Raises
    ------
    Exception
        If multiple subConnParams match the same row
    """
    import pandas as pd
    
    # Create new columns
    df = df.copy()
    df['subconn_name'] = None
    df['sec'] = None
    
    # Get population-to-cellType mapping if it exists in netParams
    pop_to_celltype = {}
    if 'popParams' in netParams:
        for pop_name, pop_params in netParams['popParams'].items():
            if 'cellType' in pop_params:
                pop_to_celltype[pop_name] = pop_params['cellType']
    
    if debug:
        print("Population to CellType mapping:")
        for pop, ct in pop_to_celltype.items():
            if pop in ['TCM', 'TC', 'PV3', 'SOM2']:
                print(f"  {pop} -> {ct}")
    
    # Helper function to check if a value matches a condition
    def value_matches_condition(value, condition):
        """Check if a value matches a condition (which can be a list or single value)"""
        if isinstance(condition, list):
            return value in condition
        else:
            return value == condition
    
    # Helper function to check if a cell matches conditions
    def matches_conds(pop_name, conds, debug_prefix=""):
        """Check if a population matches the given conditions"""
        if debug:
            print(f"{debug_prefix}Checking pop '{pop_name}' against conds: {conds}")
        
        # Check 'pop' condition - match against population name
        if 'pop' in conds:
            result = value_matches_condition(pop_name, conds['pop'])
            if debug:
                print(f"{debug_prefix}  pop condition: {conds['pop']} -> {result}")
            if not result:
                return False
        
        # Check 'cellType' condition - match against population's cellType
        if 'cellType' in conds:
            # Get cellType for this population from popParams
            cell_type = pop_to_celltype.get(pop_name)
            if debug:
                print(f"{debug_prefix}  cellType of '{pop_name}': {cell_type}")
                print(f"{debug_prefix}  cellType condition: {conds['cellType']}")
            
            if cell_type is None:
                if debug:
                    print(f"{debug_prefix}  -> No cellType found, FAIL")
                return False
            
            result = value_matches_condition(cell_type, conds['cellType'])
            if debug:
                print(f"{debug_prefix}  -> cellType match: {result}")
            if not result:
                return False
        
        if debug:
            print(f"{debug_prefix}  OVERALL: MATCH")
        return True
    
    # Iterate through dataframe rows
    for idx, row in df.iterrows():
        pre_pop = row['pre_pops']
        post_pop = row['post_pop']
        
        # Only debug specific rows
        is_debug_row = debug and pre_pop == 'TCM' and post_pop in ['PV3', 'SOM2']
        
        if is_debug_row:
            print(f"\n{'='*60}")
            print(f"Row {idx}: pre_pops='{pre_pop}', post_pop='{post_pop}'")
            print(f"{'='*60}")
        
        matched_subconns = []
        
        # Check each subConnParam
        if 'subConnParams' in netParams:
            for subconn_name, subconn_params in netParams['subConnParams'].items():
                # Check if both preConds and postConds exist
                if 'preConds' not in subconn_params or 'postConds' not in subconn_params:
                    continue
                
                # Only debug relevant subconns
                check_this = is_debug_row and subconn_name in ['TC->E', 'TCM->E', 'SOM->E', 'PV->E']
                
                if check_this:
                    print(f"\nChecking subconn '{subconn_name}':")
                
                pre_conds = subconn_params['preConds']
                post_conds = subconn_params['postConds']
                
                # Check if this subconn matches the current row
                pre_match = matches_conds(pre_pop, pre_conds, "  PRE: " if check_this else "")
                post_match = matches_conds(post_pop, post_conds, "  POST: " if check_this else "")
                
                if pre_match and post_match:
                    sec = subconn_params.get('sec', None)
                    matched_subconns.append((subconn_name, sec))
                    if check_this:
                        print(f"  >>> MATCHED! sec={sec}")
        
        if is_debug_row:
            print(f"\nTotal matches: {len(matched_subconns)}")
            if matched_subconns:
                print(f"Matched subconns: {[name for name, _ in matched_subconns]}")
        
        # Handle multiple matches
        if len(matched_subconns) > 1:
            raise Exception(
                f"Multiple subConnParams matched row {idx} "
                f"(pre_pops='{pre_pop}', post_pop='{post_pop}'): "
                f"{[name for name, _ in matched_subconns]}"
            )
        
        # Assign matched subconn if found
        if len(matched_subconns) == 1:
            df.at[idx, 'subconn_name'] = matched_subconns[0][0]
            df.at[idx, 'sec'] = matched_subconns[0][1]
    
    return df

In [12]:
def preproc_df(df):
    # rename columns
    df = df.rename(columns={
        'section_list_name': 'sec_list',
        'sections': 'secs',
        'sec': 'sec_list_sub',
        'subconn_name': 'subconn',
        'pre_pops': 'pre_pop',
    })

    # desired leading order
    first_cols = [
        'conn_name',
        'pre_pop',
        'post_pop',
        'subconn',
        'sec_list',
        'sec_list_sub',
    ]

    # keep remaining columns in any order
    other_cols = [c for c in df.columns if c not in first_cols]

    # reorder
    df = df[first_cols + other_cols]
    return df

df2 = match_subconns_to_df(net_par, df, debug=0)
df2 = preproc_df(df2)
df2


,conn_name,pre_pop,post_pop,subconn,sec_list,sec_list_sub,ynorm,secs
0,CxTh_CT5A_HTC,CT5A,HTC,None,,None,,soma
1,CxTh_CT5A_IRE,CT5A,IRE,None,,None,,soma
2,CxTh_CT5A_TC,CT5A,TC,None,,None,,soma
3,CxTh_CT5A_TI,CT5A,TI,None,,None,,soma
4,CxTh_CT5B_HTC,CT5B,HTC,None,,None,,soma
...,...,...,...,...,...,...,...,...
1540,frz_EE_PT5B_IT5B_5B,PT5Bfrz,IT5B,None,dend_all,None,"[0.667, 0.775]","Adend1, Adend2, Adend3, Bdend"
1541,frz_EE_PT5B_IT6_6,PT5Bfrz,IT6,None,dend_all,None,"[0.775, 1]","Adend1, Adend2, Adend3, Bdend"
1542,frz_EE_PT5B_ITP4_4,PT5Bfrz,ITP4,None,dend_all,None,"[0.475, 0.625]","Adend1, Adend2, Adend3, Bdend"
1543,frz_EE_PT5B_ITS4_4,PT5Bfrz,ITS4,None,dend_all,None,"[0.475, 0.625]","Adend1, Adend2, Adend3, Bdend"


In [13]:
def group_connections(df2):
    """
    Groups dataframe rows by (subconn, sec_list_sub, secs) and aggregates other columns.
    
    Parameters
    ----------
    df2 : pandas.DataFrame
        DataFrame with columns ['conn_name', 'pre_pop', 'post_pop', 'subconn', 
                                 'sec_list', 'sec_list_sub', 'ynorm', 'secs']
    
    Returns
    -------
    pandas.DataFrame
        Grouped dataframe with aggregated conn_name, pre_pop, and post_pop
    """
    import pandas as pd
    
    # Define grouping columns
    group_cols = ['subconn', 'sec_list_sub', 'secs']
    
    # Define aggregation functions for each column
    agg_dict = {
        'conn_name': lambda x: '\r\n'.join(x.astype(str).unique()),
        'pre_pop': lambda x: ', '.join(x.astype(str).unique()),
        'post_pop': lambda x: ', '.join(x.astype(str).unique()),
        'sec_list': 'first',  # assuming sec_list is the same within groups
        'ynorm': 'first'      # assuming ynorm is the same within groups
    }
    
    # Group and aggregate
    df_grouped = df2.groupby(group_cols, dropna=False).agg(agg_dict).reset_index()

    col = df_grouped.pop('sec_list')
    df_grouped.insert(2, 'sec_list', col)
    df_grouped.pop('ynorm')
    
    return df_grouped


df3 = group_connections(df2)
df3

,subconn,sec_list_sub,sec_list,secs,conn_name,pre_pop,post_pop
0,"E->E2,3,4",proximal,dend_all,"Adend1, Adend2, Adend3, Bdend",EE_CT5A_IT2_2\r\nEE_CT5A_IT3_3\r\nEE_CT5A_ITP4...,"CT5A, CT5B, CT6, IT2, IT3, IT5A, IT5B, IT6, IT...","IT2, IT3, ITP4, ITS4"
1,"E->E5,6",all,dend_all,"Adend1, Adend2, Adend3, Bdend",EE_CT5A_CT5A_5A\r\nEE_CT5A_CT5B_5B\r\nEE_CT5A_...,"CT5A, CT5B, CT6, IT2, IT3, IT5A, IT5B, IT6, IT...","CT5A, CT5B, CT6, IT5A, IT5B, IT6, PT5B"
2,E->I,all,proximal,"soma, dend",EI_CT5A_NGF1_NGF_1\r\nEI_CT5A_NGF2_NGF_2\r\nEI...,"CT5A, CT5B, CT6, IT2, IT3, IT5A, IT5B, IT6, IT...","NGF1, NGF2, NGF3, NGF4, NGF5A, NGF5B, NGF6, PV..."
3,E->I,all,proximal,"soma, rad1, ori1",EI_CT5A_VIP2_VIP_2\r\nEI_CT5A_VIP3_VIP_3\r\nEI...,"CT5A, CT5B, CT6, IT2, IT3, IT5A, IT5B, IT6, IT...","VIP2, VIP3, VIP4, VIP5A, VIP5B, VIP6"
4,NGF1->E,apic_tuft,proximal,"soma, Bdend, Adend1",IE_NGF1_NGF_CT5A_5A\r\nIE_NGF1_NGF_CT5B_5B\r\n...,NGF1,"CT5A, CT5B, CT6, IT2, IT3, IT5A, IT5B, IT6, IT..."
5,"NGF2,3,4->E2,3,4",apic_trunk,proximal,"soma, Bdend, Adend1",IE_NGF2_NGF_IT2_2\r\nIE_NGF2_NGF_IT3_3\r\nIE_N...,"NGF2, NGF3, NGF4","IT2, IT3, ITP4, ITS4"
6,"NGF2,3,4->E5,6",apic_uppertrunk,proximal,"soma, Bdend, Adend1",IE_NGF2_NGF_CT5A_5A\r\nIE_NGF2_NGF_CT5B_5B\r\n...,"NGF2, NGF3, NGF4","CT5A, CT5B, CT6, IT5A, IT5B, IT6, PT5B"
7,"NGF5,6->E5,6",apic_lowertrunk,proximal,"soma, Bdend, Adend1",IE_NGF5A_NGF_CT5A_5A\r\nIE_NGF5A_NGF_CT5B_5B\r...,"NGF5A, NGF5B, NGF6","CT5A, CT5B, CT6, IT5A, IT5B, IT6, PT5B"
8,PV->E,proximal,proximal,"soma, Bdend, Adend1",IE_PV2_PV_CT5A_5A\r\nIE_PV2_PV_CT5B_5B\r\nIE_P...,"PV2, PV3, PV4, PV5A, PV5B, PV6","CT5A, CT5B, CT6, IT2, IT3, IT5A, IT5B, IT6, IT..."
9,SOM->E,dend_all,proximal,"soma, Bdend, Adend1",IE_SOM2_SOM_CT5A_5A\r\nIE_SOM2_SOM_CT5B_5B\r\n...,"SOM2, SOM3, SOM4, SOM5A, SOM5B, SOM6","CT5A, CT5B, CT6, IT2, IT3, IT5A, IT5B, IT6, IT..."


In [14]:
from pathlib import Path
import pickle as pkl
from pprint import pprint
import sys

dirpath_root = Path().resolve().parents[1]
sys.path.append(str(dirpath_root))

import numpy as np

 ## Load data from conn pre-processing file
with open(dirpath_root / 'conn/conn.pkl', 'rb') as fid:
    connData = pkl.load(fid)
pmat = connData['pmat']
lmat = connData['lmat']
wmat = connData['wmat']
bins = connData['bins']
connDataSource = connData['connDataSource']

pprint(pmat.keys())


dict_keys(['IT2', 'IT3', 'ITP4', 'ITS4', 'IT5A', 'CT5A', 'IT5B', 'PT5B', 'CT5B', 'IT6', 'CT6', 'NGF1', 'PV2', 'SOM2', 'VIP2', 'NGF2', 'PV3', 'SOM3', 'VIP3', 'NGF3', 'PV4', 'SOM4', 'VIP4', 'NGF4', 'PV5A', 'SOM5A', 'VIP5A', 'NGF5A', 'PV5B', 'SOM5B', 'VIP5B', 'NGF5B', 'PV6', 'SOM6', 'VIP6', 'NGF6', 'TC', 'TCM', 'HTC', 'IRE', 'IREM', 'TI', 'TIM'])


In [15]:
from pprint import pformat

s = list(pmat['IREM'].keys())

print(pformat(s, width=120, compact=True))

['IREM', 'IRE', 'TC', 'HTC', 'TCM', 'TI', 'TIM']


In [9]:
dirpath_cells = dirpath_root / 'cells'
secs = {}

for fname in os.listdir(dirpath_cells):
    if not fname.endswith(".json"):
        continue
    with open(dirpath_cells / fname, "r") as f:
        data = json.load(f)
    cell_name = fname.split('_')[0]
    if 'secs' in data:
        secs[cell_name] = data['secs']

secs

{'ITP4': {'Adend1': {'geom': {'L': 307.19588441113336,
    'Ra': 70.0015514222,
    'cm': 2.74242941886,
    'diam': 1.5831889597,
    'nseg': 1,
    'pt3d': [[0, 48.4123467666, 0, 1.5831889597],
     [0, 355.6082311777334, 0, 1.5831889597]]},
   'ions': {'ca': {'e': 132.4579341637009, 'i': 5e-05, 'o': 2.0},
    'k': {'e': -104.0, 'i': 54.4, 'o': 2.5},
    'na': {'e': 42.0, 'i': 10.0, 'o': 140.0}},
   'mechs': {'cadad': {'cainf': 0.00024,
     'depth': 0.119408607923,
     'kd': 0.0,
     'kt': 0.0,
     'taur': 99.1146852282},
    'cal': {'gcalbar': 2.39132864454e-06},
    'can': {'gcanbar': 8.13137955053e-07},
    'cat': {'gcatbar': 9.29455717585e-07},
    'ih': {'ascale': 0.00320887293027,
     'ashift': 119.696272155,
     'aslope': 7.09800576233,
     'bscale': 0.285307415701,
     'bslope': 23.2995848558,
     'gbar': 3.3176340367e-05},
    'kBK': {'caPh': 0.002,
     'caPk': 1.0,
     'caPmax': 1.0,
     'caPmin': 0.0,
     'caVhh': 0.002,
     'caVhmax': 155.67,
     'caVhmin':

In [18]:
import pandas as pd

df = []
secs_used = ['Bdend', 'soma', 'Adend1', 'Adend2', 'Adend3']

for pop, pop_secs in secs.items():
    entry = {'pop': pop}
    nfound = 0
    for sec_used in secs_used:
        if sec_used in pop_secs:
            entry[sec_used] = pop_secs[sec_used]['geom']['L']
            nfound += 1
        else:
            entry[sec_used] = None
    if nfound == len(secs_used):
        df.append(entry)

df = pd.DataFrame(df)
df.round(0)

,pop,Bdend,soma,Adend1,Adend2,Adend3
0,ITP4,164.0,48.0,307.0,307.0,307.0
1,PT5B,290.0,48.0,448.0,448.0,448.0
2,IT3,97.0,48.0,152.0,152.0,152.0
3,IT2,97.0,48.0,11.0,11.0,11.0
4,IT6,398.0,48.0,109.0,109.0,109.0
5,IT5A,212.0,48.0,398.0,398.0,398.0
6,ITS4,164.0,15.0,181.0,181.0,181.0
7,IT5B,290.0,48.0,448.0,448.0,448.0
8,CT5B,398.0,48.0,209.0,209.0,209.0
9,CT5A,398.0,48.0,209.0,209.0,209.0
